## **Agentic Sales Pipeline**

# Setup and load yaml files

Remember yaml files control the behavior of the multi agent system

In [44]:
from dotenv import load_dotenv
load_dotenv()

import os
import yaml
from crewai import Agent, Crew, Task, LLM

In [45]:
llm_model = LLM(model="gemini/gemini-3.1-flash-lite", temperature=0.7)  # check the recent model

In [46]:
files = {
    'lead_agents': 'config/lead_qualification_agents.yaml',
    'lead_tasks': 'config/lead_qualification_tasks.yaml',
    'email_agents': 'config/email_engagement_agents.yaml',
    'email_tasks': 'config/email_engagement_tasks.yaml'
}

configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

lead_agents_config = configs['lead_agents']
lead_tasks_config = configs['lead_tasks']
email_agents_config = configs['email_agents']
email_tasks_config = configs['email_tasks']

# Pydantic Structure Schemas 

- It is used to put out strucutured outputs from Crew
- Think about all the input and output in the workflow and make the here

In [47]:
from pydantic import BaseModel, Field
from typing import Dict, Optional, List, Set, Tuple

class LeadPersonalInfo(BaseModel):
    name: str = Field(..., description="The full name of the lead.")
    job_title: str = Field(..., description="The job title of the lead.")
    role_relevance: int = Field(..., ge=0, le=10, description="A score representing how relevant the lead's role is to the decision-making process (0-10).")
    professional_background: Optional[str] = Field(..., description="A brief description of the lead's professional background.")

class CompanyInfo(BaseModel):
    company_name: str = Field(..., description="The name of the company the lead works for.")
    industry: str = Field(..., description="The industry in which the company operates.")
    company_size: int = Field(..., description="The size of the company in terms of employee count.")
    revenue: Optional[float] = Field(None, description="The annual revenue of the company, if available.")
    market_presence: int = Field(..., ge=0, le=10, description="A score representing the company's market presence (0-10).")

class LeadScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="The final score assigned to the lead (0-100).")
    scoring_criteria: List[str] = Field(..., description="The criteria used to determine the lead's score.")
    validation_notes: Optional[str] = Field(None, description="Any notes regarding the validation of the lead score.")

class LeadScoringResult(BaseModel):
    personal_info: LeadPersonalInfo = Field(..., description="Personal information about the lead.")
    company_info: CompanyInfo = Field(..., description="Information about the lead's company.")
    lead_score: LeadScore = Field(..., description="The calculated score and related information for the lead.")

# Importing Tools

Here we will be importing tools directly but we can build tools separately

In [48]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# Orchestration

Here we make *agents* and then connect them to *tasks*, finally at the end we combine all agents into a *Crew*

## Lead Qualification Crew

### Agents

In [49]:
lead_data_agent = Agent(
  config=lead_agents_config['lead_data_agent'],    # you need to connect the most specific agent here(look the yaml file)
  tools=[SerperDevTool(), ScrapeWebsiteTool()],
  llm=llm_model
)

cultural_fit_agent = Agent(
  config=lead_agents_config['cultural_fit_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()],
  llm=llm_model
)

scoring_validation_agent = Agent(
  config=lead_agents_config['scoring_validation_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()],
  llm=llm_model
)

### Tasks

In [50]:
lead_data_task= Task(
    config=lead_tasks_config['lead_data_collection'],
    agent=lead_data_agent
)

cultural_fit_task = Task(
  config=lead_tasks_config['cultural_fit_analysis'],
  agent=cultural_fit_agent
)

scoring_validation_task = Task(
  config=lead_tasks_config['lead_scoring_and_validation'],
  agent=scoring_validation_agent,
  context=[lead_data_task, cultural_fit_task],
  output_pydantic=LeadScoringResult
)

### Crew

In [51]:
lead_scoring_crew = Crew(
    agents=[
        lead_data_agent,
        cultural_fit_agent,
        scoring_validation_agent
    ],
    tasks=[
        lead_data_task,
        cultural_fit_task,
        scoring_validation_task
    ],
    verbose=True
)

## Email Engagement Crew

### Agents

In [52]:
email_content_specialist = Agent(
  config=email_agents_config['email_content_specialist'],
  llm=llm_model
)

engagement_strategist = Agent(
  config=email_agents_config['engagement_strategist'],
  llm=llm_model
)

### Tasks

In [53]:
email_drafting = Task(
  config=email_tasks_config['email_drafting'],
  agent=email_content_specialist
)

engagement_optimization = Task(
  config=email_tasks_config['engagement_optimization'],
  agent=engagement_strategist
)


### Crew

In [54]:
email_writing_crew = Crew(
    agents=[
        email_content_specialist,
        engagement_strategist
    ],
    tasks=[
        email_drafting,
        engagement_optimization
    ],
    verbose=True
)

## Flow

It combines all the crews

In [55]:
from crewai import Flow
from crewai.flow.flow import listen, start

class SalesPipeline(Flow):
    def __init__(self):
        super().__init__(tracing=True)  # Enable tracing for the flow

    @start()
    def fetch_leads(self):
        # Pull our leads from the database
        leads = [
            {
                "lead_data": {
                    "name": "João Moura",
                    "job_title": "Director of Engineering",
                    "company": "Clearbit",
                    "email": "joao@clearbit.com",
                    "use_case": "Using AI Agent to do better data enrichment."
                },
            },
        ]
        return leads

    @listen(fetch_leads)
    def score_leads(self, leads):
        scores = lead_scoring_crew.kickoff_for_each(leads)
        self.state["score_crews_results"] = scores
        return scores

    @listen(score_leads)
    def store_leads_score(self, scores):
        # Here we would store the scores in the database
        return scores

    @listen(score_leads)
    def filter_leads(self, scores):
        score_list = [score for score in scores if score['lead_score'].score > 70]
        print("score_list output: ", score_list)
        return score_list
    
    @listen(filter_leads)
    def write_email(self, leads):
        scored_leads = [lead.to_dict() for lead in leads]
        emails = email_writing_crew.kickoff_for_each(scored_leads)
        return emails

    @listen(write_email)
    def send_email(self, emails):
        # Here we would send the emails to the leads
        return emails

flow = SalesPipeline()

# Kickoff Flow

In [56]:
# flow.plot()

In [57]:
emails = flow.kickoff()
print("Final email outputs: ", emails)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: SalesPipeline                                                                                            │
│  ID: a80d4349-c46b-4e0d-b115-b384f77146e7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: SalesPipeline                                                                                            │
│  ID: a80d4349-c46b-4e0d-b115-b384f77146e7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: a80d4349-c46b-4e0d-b115-b384f77146e7

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: fetch_leads                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: fetch_leads                                                                                            │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: send_email                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: send_email                                                                                             │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: SalesPipeline                                                                                            │
│  ID: a80d4349-c46b-4e0d-b115-b384f77146e7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final email outputs:  [CrewOutput(raw='To maximize conversion, an outreach strategy must treat the prospect not as a target, but as a peer. The "Plug-and-Play" email you have is excellent for a warm-up; now, we must wrap it in a **14-day high-intent cadence.**\n\nBelow is your **Outreach Sequence Playbook.**\n\n---\n\n### The 14-Day "Relationship-First" Cadence\n\nThis sequence avoids the "spam" trap by rotating channels and slowly increasing the value provided.\n\n| Day | Channel | Action | Objective |\n| :--- | :--- | :--- | :--- |\n| **Day 1** | Email | The "Plug-and-Play" Email | Initial value-add/Pattern interrupt. |\n| **Day 2** | LinkedIn | Connection Request (No Note) | Put a face to the name. |\n| **Day 3** | LinkedIn | Like/Comment on their recent post | Establish familiarity before the follow-up. |\n| **Day 4** | Email | Follow-up (The "Bump") | Keep it top of mind. |\n| **Day 8** | LinkedIn | Direct Message (The "Value Add") | Provide a resource (case study/article). |\n| *